# NutriRisk AI — Análisis exploratorio de los datos sintéticos

Este cuaderno reproduce el análisis exploratorio de la encuesta sintética de *República de Andara* y verifica la
coherencia de los z-scores OMS calculados por el motor del proyecto.

> **Datos sintéticos**: no representan prevalencias reales. Semilla fija 42.

Requisitos: `pandas`, `numpy`, `matplotlib` (incluidos en el entorno del proyecto). Ejecute desde la carpeta `notebooks/`
o ajuste la ruta de `data/synthetic/children_synthetic.csv`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

from app.services.anthropometry import compute_indicators, self_check  # noqa: E402

frame = pd.read_csv(ROOT / "data" / "synthetic" / "children_synthetic.csv")
print(frame.shape)
frame.head()

In [ ]:
print("Territorios:", frame["territory_code"].nunique(), "| Regiones:", frame["region"].nunique())
print("Años:", sorted(frame["period_year"].unique()))
print("Niños por año:")
print(frame.groupby("period_year").size())
print("\nNulos por columna (top 5):")
print(frame.isna().mean().sort_values(ascending=False).head())

In [ ]:
results = [
    compute_indicators(
        sex=row.sex,
        age_months=row.age_months,
        weight_kg=row.weight_kg,
        height_cm=row.height_cm,
        edema=row.edema,
    )
    for row in frame.itertuples(index=False)
]
scored = frame.copy()
scored["haz"] = [item.haz for item in results]
scored["waz"] = [item.waz for item in results]
scored["whz"] = [item.whz for item in results]
scored["stunting"] = [item.stunting for item in results]
scored["wasting"] = [item.wasting for item in results]
scored["underweight"] = [item.underweight for item in results]
print(f"Retraso: {scored.stunting.mean():.1%} | Emaciación: {scored.wasting.mean():.1%} | Bajo peso: {scored.underweight.mean():.1%}")
print("Autochequeo OMS:", self_check())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(scored["haz"], bins=40, color="#0f766e", alpha=0.85)
axes[0].axvline(-2, color="#dc2626", linestyle="--", label="-2 DE")
axes[0].axvline(-3, color="#7f1d1d", linestyle=":", label="-3 DE")
axes[0].set_title("Distribución de HAZ")
axes[0].set_xlabel("z-score")
axes[0].legend()

by_region = scored.groupby("region")[["stunting", "wasting", "underweight"]].mean().sort_values("stunting")
by_region.plot(kind="barh", ax=axes[1], color=["#0f766e", "#f59e0b", "#0e7490"])
axes[1].set_title("Prevalencias por región (sintéticas)")
axes[1].set_xlabel("proporción")
plt.tight_layout()
plt.show()

In [ ]:
determinants = [
    "wealth_quintile", "maternal_education", "water_source", "sanitation",
    "food_insecurity_score", "ndvi", "altitude_m", "urban_rural",
]
encoded = scored[determinants + ["haz"]].copy()
for column in ("maternal_education", "water_source", "sanitation", "urban_rural"):
    encoded[column] = encoded[column].astype("category").cat.codes
encoded.corr()["haz"].sort_values().to_frame("correlación con HAZ")

## Conclusiones del EDA

1. La media de HAZ es negativa y la prevalencia de retraso se sitúa en el rango esperado para un país de renta media-baja,
   con heterogeneidad territorial marcada (Norte y Sur con mayor carga).
2. La riqueza del hogar, la educación materna y el acceso a agua/saneamiento muestran la asociación esperada con HAZ.
3. Las variables ambientales (NDVI, altitud, precipitación) aportan señal a nivel territorial; por eso el modelo híbrido
   incorpora rezagos espaciales y el mapa agrega el riesgo por territorio.
4. El autochequeo confirma que el motor LMS reproduce los puntos de corte publicados por la OMS.

Para el análisis de modelos, explicabilidad y equidad use el tablero y los módulos *Modelos*, *Explicabilidad* y *Equidad*.